# SpaceNet 8 Evaluation Notebook

**What this notebook does:**
1. Installs and verifies dependencies
2. Builds an SN8 manifest (one row per building crop, with pixel coordinates)
3. Runs your trained ResNet18 model on SN8 for cross-dataset evaluation
4. Reports macro-F1 for your paper

**Key fixes vs original code:**
- `build_sn8_manifest`: uses `label_image_mapping.csv` + rasterio lon/lat→pixel conversion
- Flood label: checks `flooded == 'yes'` (actual SN8 value, not True/False)
- Building filter: uses Polygon/MultiPolygon geometry (not `building=='yes'` which is None in data)
- Includes BOTH flooded and non-flooded buildings (not just flooded)
- Inference: collapses 4-class xBD predictions → binary flood for SN8 evaluation
- Device: cuda → mps → cpu auto-detection

**Place your `gold_pipeline.py` in the same folder as this notebook.**

## Cell 1 — Install dependencies

In [1]:
# Run once — restart kernel after if prompted
!pip install "osmnx>=2.0" scikit-learn shapely networkx pillow rasterio --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2 — Imports

In [2]:
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

print('Imports OK')

Imports OK


## Cell 3 — OSMnx road enrichment helpers (unchanged, correct)

In [3]:
def extract_building_coords(label_json_path: Path, damage_filter=None):
    """Extract (lng, lat) centroids of buildings from xView2 label JSON."""
    with open(label_json_path) as f:
        data = json.load(f)
    buildings = []
    features = data.get('features', {}).get('lng_lat', [])
    for feat in features:
        props = feat.get('properties', {})
        subtype = props.get('subtype', 'unknown')
        if damage_filter and subtype not in damage_filter:
            continue
        wkt = feat.get('wkt', '')
        coords = _wkt_to_centroid(wkt)
        if coords:
            buildings.append({'uid': props.get('uid', ''), 'subtype': subtype,
                               'lng': coords[0], 'lat': coords[1]})
    return buildings


def _wkt_to_centroid(wkt: str):
    try:
        coords_str = wkt.replace('POLYGON ((', '').replace('))', '').strip()
        pairs = [p.strip().split() for p in coords_str.split(',')]
        lngs = [float(p[0]) for p in pairs if len(p) == 2]
        lats = [float(p[1]) for p in pairs if len(p) == 2]
        return (sum(lngs) / len(lngs), sum(lats) / len(lats))
    except Exception:
        return None


def dbscan_hotspots(buildings: list, eps_deg: float = 0.001, min_samples: int = 3):
    from sklearn.cluster import DBSCAN
    if len(buildings) < min_samples:
        if buildings:
            lngs = [b['lng'] for b in buildings]
            lats = [b['lat'] for b in buildings]
            return [{'cluster_id': 0, 'n_buildings': len(buildings),
                     'centroid_lng': round(sum(lngs)/len(lngs), 6),
                     'centroid_lat': round(sum(lats)/len(lats), 6),
                     'bbox': [min(lngs), min(lats), max(lngs), max(lats)]}]
        return []
    coords = np.array([[b['lng'], b['lat']] for b in buildings])
    labels = DBSCAN(eps=eps_deg, min_samples=min_samples, metric='euclidean').fit_predict(coords)
    hotspots = []
    for cluster_id in set(labels):
        if cluster_id == -1:
            continue
        mask = (labels == cluster_id)
        cluster_lngs = coords[mask, 0]
        cluster_lats = coords[mask, 1]
        hotspots.append({
            'cluster_id':   int(cluster_id),
            'n_buildings':  int(mask.sum()),
            'centroid_lng': round(float(cluster_lngs.mean()), 6),
            'centroid_lat': round(float(cluster_lats.mean()), 6),
            'bbox': [round(float(cluster_lngs.min()), 6), round(float(cluster_lats.min()), 6),
                     round(float(cluster_lngs.max()), 6), round(float(cluster_lats.max()), 6)],
        })
    hotspots.sort(key=lambda h: h['n_buildings'], reverse=True)
    return hotspots


def get_road_density(lat: float, lng: float, radius_m: int = 500) -> dict:
    """OSMnx road density — FIX 1: no ox.basic_stats(), FIX 2: rate limiting."""
    try:
        import osmnx as ox
        G = ox.graph_from_point((lat, lng), dist=radius_m, network_type='drive', simplify=True)
        total_length_m = sum(d.get('length', 0) for u, v, k, d in G.edges(keys=True, data=True))
        n_edges = G.number_of_edges()
        G_u = ox.convert.to_undirected(G)
        n_intersections = sum(1 for node, deg in G_u.degree() if deg >= 3)
        area_km2 = np.pi * (radius_m / 1000) ** 2
        density = (total_length_m / 1000) / area_km2 if area_km2 > 0 else 0
        risk = 'low' if density > 8 else 'moderate' if density > 3 else 'high' if density > 0.5 else 'critical'
        return {'road_length_m': round(total_length_m, 0), 'n_road_segments': n_edges,
                'n_intersections': n_intersections, 'road_density_km_km2': round(density, 2),
                'access_risk': risk, 'query_radius_m': radius_m}
    except Exception as e:
        return {'road_length_m': 0, 'n_road_segments': 0, 'n_intersections': 0,
                'road_density_km_km2': 0.0, 'access_risk': 'unknown',
                'query_radius_m': radius_m, 'error': str(e)}


def enrich_hotspots_with_roads(hotspots: list, radius_m: int = 500) -> list:
    enriched = []
    for i, h in enumerate(hotspots):
        road_info = get_road_density(h['centroid_lat'], h['centroid_lng'], radius_m)
        enriched.append({**h, **road_info})
        print(f"  Hotspot {h['cluster_id']}: {h['n_buildings']} bldgs, "
              f"road_density={road_info['road_density_km_km2']:.1f}, access_risk={road_info['access_risk']}")
        if i < len(hotspots) - 1:
            time.sleep(1.0)
    return enriched


SEVERE_CLASSES = {'major-damage', 'destroyed'}

def compute_priority(severe_ratio_pct):
    if severe_ratio_pct >= 50:   return 'Immediate response priority — majority of structures severely damaged.'
    elif severe_ratio_pct >= 20: return 'Elevated response priority — significant structural damage.'
    elif severe_ratio_pct >= 5:  return 'Moderate response priority — limited severe damage.'
    else:                        return 'Low response priority — minimal severe damage detected.'


def build_gold_scene_json(scene_id, label_dir, use_osmnx=True, road_radius_m=500,
                           dbscan_eps=0.001, dbscan_min=3):
    label_path = label_dir / f'{scene_id}_post_disaster.json'
    if not label_path.exists():
        raise FileNotFoundError(f'Label not found: {label_path}')
    all_buildings  = extract_building_coords(label_path, damage_filter=None)
    severe_bldgs   = [b for b in all_buildings if b['subtype'] in SEVERE_CLASSES]
    damage_counter = Counter(b['subtype'] for b in all_buildings)
    total          = len(all_buildings)
    severe_count   = sum(damage_counter.get(c, 0) for c in SEVERE_CLASSES)
    severe_ratio   = severe_count / total if total > 0 else 0.0
    hotspots = dbscan_hotspots(severe_bldgs, eps_deg=dbscan_eps, min_samples=dbscan_min)
    if use_osmnx and hotspots:
        print(f'Enriching {len(hotspots)} hotspot(s) with OSMnx road data...')
        hotspots = enrich_hotspots_with_roads(hotspots, radius_m=road_radius_m)
    prefix = scene_id.split('_')[0].lower()
    if 'hurricane' in prefix:              disaster = 'hurricane'
    elif 'flood' in prefix:                disaster = 'flood'
    elif 'fire' in prefix:                 disaster = 'fire'
    elif 'earthquake' in prefix:           disaster = 'earthquake'
    elif 'tsunami' in prefix:              disaster = 'tsunami'
    elif 'volcano' in prefix:              disaster = 'volcano'
    else:                                  disaster = 'unknown'
    return {
        'scene_id': scene_id, 'disaster_type': disaster, 'total_buildings': total,
        'damage_counts': {k: int(damage_counter.get(k, 0)) for k in
                          ['no-damage','minor-damage','major-damage','destroyed']},
        'severe_count': severe_count, 'severe_ratio': round(severe_ratio, 4),
        'severe_ratio_pct': round(severe_ratio * 100, 1),
        'hotspot_count': len(hotspots), 'hotspots': hotspots,
        'priority': compute_priority(severe_ratio * 100),
    }

print('Step 1-3 helpers loaded OK')

Step 1-3 helpers loaded OK


## Cell 4 — SN8 label mapping

In [4]:
def map_sn8_label_to_xbd(sn8_label: str) -> str:
    """
    Map SN8 flood label → xBD 4-class schema.
    Confirmed actual SN8 values: None (not flooded) and 'yes' (flooded).
    """
    mapping = {
        # Confirmed actual SN8 values
        'yes':          'major-damage',   # flooded building/road
        'no':           'no-damage',      # explicitly not flooded
        'none':         'no-damage',      # Python None → str
        # Boolean variants (just in case)
        'true':         'major-damage',
        'false':        'no-damage',
        # Verbose variants
        'flooded':      'major-damage',
        'non-flooded':  'no-damage',
        'nonflooded':   'no-damage',
        # xBD passthrough
        'no-damage':    'no-damage',
        'minor-damage': 'minor-damage',
        'major-damage': 'major-damage',
        'destroyed':    'destroyed',
    }
    normalised = str(sn8_label).lower().strip().replace(' ', '-')
    return mapping.get(normalised, 'no-damage')

print('Label mapping loaded OK')

Label mapping loaded OK


## Cell 5 — `build_sn8_manifest` (best-of-both: rasterio pixel coords + correct filters)

In [5]:
def build_sn8_manifest(
    sn8_data_dir: Path,
    output_csv: Path,
    pad: int = 32,
    min_size: int = 96,
) -> pd.DataFrame:
    """
    Build a per-building-crop manifest from SpaceNet 8 data.

    Best-of-both approach:
    - Uses label_image_mapping.csv for reliable tile→image joining (not glob guessing)
    - Uses rasterio to convert lon/lat polygon coords → pixel bounding boxes
    - Filters to Polygon/MultiPolygon geometry (buildings) only
      NOTE: Does NOT filter on building=='yes' because in Germany AOI
      that property is None for all features — geometry type is the
      reliable discriminator.
    - Includes BOTH flooded and non-flooded buildings (both are needed
      for a valid binary evaluation — skipping non-flooded would give
      a trivially high recall with no precision signal)
    - Flood label: flooded=='yes' (confirmed actual SN8 string value)

    Output columns:
      aoi, tile_id, building_index, pre_path, post_path,
      x1, y1, x2, y2, center_x, center_y,
      sn8_label, label (xBD mapped), flooded (0/1)
    """
    import rasterio

    def lonlat_to_pixel_bbox(coords_rings, dataset, pad, min_size, W, H):
        """Convert GeoJSON polygon lon/lat rings to padded pixel bounding box."""
        xs, ys = [], []
        for ring in coords_rings:
            for pt in ring:
                if len(pt) >= 2:
                    try:
                        row, col = dataset.index(pt[0], pt[1])  # lon, lat → row, col
                        xs.append(col)
                        ys.append(row)
                    except Exception:
                        pass
        if not xs or not ys:
            return None

        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)

        x1 = max(0, int(xmin) - pad)
        y1 = max(0, int(ymin) - pad)
        x2 = min(W, int(xmax) + pad)
        y2 = min(H, int(ymax) + pad)

        # Enforce minimum crop size
        if (x2 - x1) < min_size or (y2 - y1) < min_size:
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            half   = min_size // 2
            x1 = max(0, cx - half); x2 = min(W, cx + half)
            y1 = max(0, cy - half); y2 = min(H, cy + half)

        return x1, y1, x2, y2

    records          = []
    raw_label_counts = Counter()
    skipped_oob      = 0
    skipped_no_geom  = 0

    aoi_dirs = [d for d in sn8_data_dir.iterdir() if d.is_dir()]
    if not aoi_dirs:
        raise FileNotFoundError(f'No AOI subdirectories found in {sn8_data_dir}')

    for aoi_dir in aoi_dirs:
        mapping_files = list(aoi_dir.glob('*_label_image_mapping.csv'))
        if not mapping_files:
            print(f'  Skipping {aoi_dir.name}: no *_label_image_mapping.csv')
            continue

        mapping  = pd.read_csv(mapping_files[0])
        pre_dir  = aoi_dir / 'PRE-event'
        post_dir = aoi_dir / 'POST-event'
        ann_dir  = aoi_dir / 'annotations'

        for folder in [pre_dir, post_dir, ann_dir]:
            if not folder.exists():
                print(f'  Skipping {aoi_dir.name}: {folder.name}/ missing')
                continue

        print(f'  Processing {aoi_dir.name}: {len(mapping)} tiles...')

        for _, row in mapping.iterrows():
            geojson_name = str(row['label'])
            pre_name     = str(row['pre-event image'])
            post_name    = str(row['post-event image 1'])
            tile_id      = Path(geojson_name).stem

            geojson_path = ann_dir  / geojson_name
            pre_path     = pre_dir  / pre_name
            post_path    = post_dir / post_name

            if not geojson_path.exists() or not pre_path.exists():
                continue
            if not post_path.exists():
                candidates = list(post_dir.glob(f'*{tile_id}*'))
                if not candidates:
                    continue
                post_path = candidates[0]

            try:
                with open(geojson_path, encoding='utf-8') as f:
                    gj = json.load(f)
            except Exception as e:
                print(f'    Cannot read {geojson_name}: {e}')
                continue

            features = gj.get('features', [])
            if not features:
                continue

            # Open raster once per tile for coordinate transform
            try:
                ds = rasterio.open(pre_path)
                W, H = ds.width, ds.height
            except Exception as e:
                print(f'    Cannot open raster {pre_name}: {e}')
                continue

            building_index = 0
            for feat in features:
                geom = feat.get('geometry', {})
                if not geom:
                    skipped_no_geom += 1
                    continue

                geom_type = geom.get('type', '')

                # ── BUILDING FILTER ──────────────────────────────────────────
                # Use geometry type — NOT building=='yes' which is None in
                # Germany AOI. Polygons = buildings, LineStrings = roads.
                if geom_type == 'Polygon':
                    coord_rings = geom.get('coordinates', [])
                elif geom_type == 'MultiPolygon':
                    # Flatten MultiPolygon rings
                    coord_rings = [ring for poly in geom.get('coordinates', []) for ring in poly]
                else:
                    continue  # Skip roads (LineString/MultiLineString)

                props = feat.get('properties', {})

                # ── FLOOD LABEL ──────────────────────────────────────────────
                # Confirmed SN8 Germany values: None (not flooded), 'yes' (flooded)
                raw_flood = props.get('flooded', props.get('Flooded', None))
                is_flooded = (raw_flood == 'yes')
                sn8_label  = 'yes' if is_flooded else 'no'
                xbd_label  = map_sn8_label_to_xbd(sn8_label)
                raw_label_counts[sn8_label] += 1

                # ── PIXEL BBOX ───────────────────────────────────────────────
                bbox = lonlat_to_pixel_bbox(coord_rings, ds, pad, min_size, W, H)
                if bbox is None:
                    skipped_oob += 1
                    continue
                x1, y1, x2, y2 = bbox

                records.append({
                    'aoi':            aoi_dir.name,
                    'tile_id':        tile_id,
                    'building_index': building_index,
                    'pre_path':       str(pre_path),
                    'post_path':      str(post_path),
                    'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                    'center_x':       (x1 + x2) / 2.0,
                    'center_y':       (y1 + y2) / 2.0,
                    'sn8_label':      sn8_label,
                    'label':          xbd_label,
                    'flooded':        int(is_flooded),
                })
                building_index += 1

            ds.close()

    if not records:
        raise ValueError(
            'Manifest is empty. No building crops were extracted.\n'
            f'Skipped {skipped_oob} features (out-of-bounds), '
            f'{skipped_no_geom} (no geometry).'
        )

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)

    print(f'\nSN8 manifest built:')
    print(f'  Total building crops : {len(df)}')
    print(f'  Flooded buildings    : {df["flooded"].sum()}')
    print(f'  Non-flooded buildings: {(df["flooded"] == 0).sum()}')
    print(f'  Tiles covered        : {df["tile_id"].nunique()} / 202')
    print(f'  AOIs                 : {df["aoi"].nunique()}')
    print(f'  Skipped (no coords)  : {skipped_oob}')
    print(f'  Raw SN8 label counts : {dict(raw_label_counts)}')
    print(f'  xBD label dist       :')
    print(df['label'].value_counts().to_string())
    print(f'\n  Saved → {output_csv}')
    return df

print('build_sn8_manifest loaded OK')

build_sn8_manifest loaded OK


## Cell 6 — `run_inference_on_spacenet8_crops`

In [6]:
def run_inference_on_spacenet8_crops(
    sn8_manifest_csv: str,
    checkpoint_path: str,
    output_csv: str,
    use_9ch: bool = True,
    img_size: int = 224,
    batch_size: int = 32,
) -> float:
    """
    Run trained ResNet18 on SN8 building crops for cross-dataset evaluation.

    The model outputs 4-class xBD predictions which are collapsed to binary:
      no-damage / minor-damage → non-flooded (0)
      major-damage / destroyed → flooded (1)

    Metric: macro-F1 (flooded vs non-flooded) — reported in paper.
    """
    import torch
    import torch.nn as nn
    import torchvision.models as models
    import torchvision.transforms as T
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image
    from sklearn.metrics import f1_score, classification_report, confusion_matrix

    # FIX 6: Proper device detection
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(f'Device: {device}')

    channels = 9 if use_9ch else 6

    class SN8CropDataset(Dataset):
        def __init__(self, df, transform, use_9ch=True):
            self.df        = df.reset_index(drop=True)
            self.transform = transform
            self.use_9ch   = use_9ch

        def __len__(self): return len(self.df)

        def __getitem__(self, idx):
            r = self.df.iloc[idx]
            x1, y1, x2, y2 = int(r['x1']), int(r['y1']), int(r['x2']), int(r['y2'])
            try:
                pre  = Image.open(r['pre_path']).convert('RGB').crop((x1, y1, x2, y2))
                post = Image.open(r['post_path']).convert('RGB').crop((x1, y1, x2, y2))
            except Exception as e:
                 raise RuntimeError(f"Error at idx {idx}: {e}")

            pre_t  = self.transform(pre)
            post_t = self.transform(post)
            if self.use_9ch:
                diff_t = torch.abs(post_t - pre_t)
                x = torch.cat([pre_t, post_t, diff_t], dim=0)  # [9, H, W]
                mean = torch.tensor([0.485,0.456,0.406] * 3,dtype=torch.float32).view(9,1,1)
                std=torch.tensor([0.229,0.224,0.225] * 3,dtype=torch.float32).view(9,1,1)
            else:
                x = torch.cat([pre_t, post_t], dim=0)
                mean=torch.tensor([0.485,0.456,0.406] * 2,dtype=torch.float32).view(6,1,1)
                std=torch.tensor([0.229,0.224,0.225] * 2,dtype=torch.float32).view(6,1,1)
            
            x= (x - mean) / (std)
            y = torch.tensor(int(r['flooded']), dtype=torch.long)
            return x, y

    df = pd.read_csv(sn8_manifest_csv)
    print(f'Manifest loaded: {len(df)} building crops')
    print(f'  Flooded    : {df["flooded"].sum()}')
    print(f'  Non-flooded: {(df["flooded"]==0).sum()}')

    tfm = T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        
    ])

    ds  = SN8CropDataset(df, tfm, use_9ch=use_9ch)
    ldr = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

    # Load model
    m = models.resnet18(weights=None)
    m.conv1 = nn.Conv2d(channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    m.fc    = nn.Linear(m.fc.in_features, 4)
    m.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    m = m.to(device).eval()
    print(f'Model loaded: {checkpoint_path}')

    all_preds_4class = []
    all_labels       = []

    with torch.no_grad():
        for i, (x, y) in enumerate(ldr):
            preds = m(x.to(device)).argmax(1).cpu().numpy()
            all_preds_4class.extend(preds.tolist())
            all_labels.extend(y.numpy().tolist())
            if (i + 1) % 20 == 0:
                print(f'  {(i+1)*batch_size}/{len(df)} crops processed...')

    # Collapse 4-class → binary: 0,1 → non-flooded; 2,3 → flooded
    preds_binary = [1 if p >= 2 else 0 for p in all_preds_4class]
    labels       = [int(l) for l in all_labels]

    macro_f1   = f1_score(labels, preds_binary, average='macro')
    flooded_f1 = f1_score(labels, preds_binary, pos_label=1, average='binary')

    print(f'\n{"="*55}')
    print('SpaceNet 8 Cross-Dataset Evaluation (Building Crops)')
    print(f'{"="*55}')
    print(classification_report(labels, preds_binary,
                                target_names=['non-flooded', 'flooded'], digits=3))
    print(f'Macro-F1   : {macro_f1:.4f}')
    print(f'Flooded-F1 : {flooded_f1:.4f}')
    print(f'\nConfusion matrix (rows=true, cols=pred):')
    print(confusion_matrix(labels, preds_binary))
    print(f'\nExpected range: 0.40–0.65 for cross-dataset generalisation')
    print('Report this value in your paper regardless — it is a valid finding.')

    xbd_classes = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
    df['pred_xbd_label'] = [xbd_classes[p] for p in all_preds_4class]
    df['pred_flooded']   = preds_binary
    df['true_flooded']   = labels
    df['correct']        = [int(p == t) for p, t in zip(preds_binary, labels)]
    df.to_csv(output_csv, index=False)
    print(f'\nSaved per-crop predictions → {output_csv}')
    return macro_f1

print('run_inference_on_spacenet8_crops loaded OK')

run_inference_on_spacenet8_crops loaded OK


## Cell 7 — Verify OSMnx (optional, needed for xView2 Gold pipeline only)

In [7]:
def verify_osmnx():
    try:
        import osmnx as ox
        version = ox.__version__
        if int(version.split('.')[0]) < 2:
            print(f'⚠️  OSMnx {version} — upgrade: pip install osmnx>=2.0')
            return False
        print(f'✅  OSMnx {version}')
    except ImportError:
        print('❌  OSMnx not installed')
        return False
    result = get_road_density(lat=29.749, lng=-95.358, radius_m=500)
    if 'error' in result:
        print(f'❌  Overpass failed: {result["error"]}')
        return False
    print(f'✅  Overpass OK — road_density={result["road_density_km_km2"]}, access_risk={result["access_risk"]}')
    return True

# Uncomment to run:
# verify_osmnx()

---
## ▶ RUN FROM HERE
---
## Cell 8 — Build SN8 manifest

Produces `sn8_manifest.csv` with one row per building crop.

In [9]:
sn8_df = build_sn8_manifest(
    sn8_data_dir=Path('./spacenet8'),
    output_csv=Path('sn8_manifest.csv'),
    pad=32,
    min_size=96,
)
sn8_df.head()

  Processing Germany_Training_Public: 202 tiles...
  Skipping tarballs: no *_label_image_mapping.csv

SN8 manifest built:
  Total building crops : 3986
  Flooded buildings    : 1250
  Non-flooded buildings: 2736
  Tiles covered        : 129 / 202
  AOIs                 : 1
  Skipped (no coords)  : 0
  Raw SN8 label counts : {'yes': 1250, 'no': 2736}
  xBD label dist       :
label
no-damage       2736
major-damage    1250

  Saved → sn8_manifest.csv


,aoi,tile_id,building_index,pre_path,post_path,x1,y1,x2,y2,center_x,center_y,sn8_label,label,flooded
0,Germany_Training_Public,0_41_59,0,spacenet8\Germany_Training_Public\PRE-event\10...,spacenet8\Germany_Training_Public\POST-event\1...,1062,0,1158,89,1110.0,44.5,yes,major-damage,1
1,Germany_Training_Public,0_41_59,1,spacenet8\Germany_Training_Public\PRE-event\10...,spacenet8\Germany_Training_Public\POST-event\1...,988,13,1121,128,1054.5,70.5,yes,major-damage,1
2,Germany_Training_Public,0_41_59,2,spacenet8\Germany_Training_Public\PRE-event\10...,spacenet8\Germany_Training_Public\POST-event\1...,969,71,1097,179,1033.0,125.0,yes,major-damage,1
3,Germany_Training_Public,0_41_59,3,spacenet8\Germany_Training_Public\PRE-event\10...,spacenet8\Germany_Training_Public\POST-event\1...,941,120,1083,244,1012.0,182.0,yes,major-damage,1
4,Germany_Training_Public,0_41_59,4,spacenet8\Germany_Training_Public\PRE-event\10...,spacenet8\Germany_Training_Public\POST-event\1...,925,186,1069,330,997.0,258.0,yes,major-damage,1


## Cell 9 — Quick sanity check on the manifest

In [10]:
df = pd.read_csv('sn8_manifest.csv')

print(f'Total crops      : {len(df)}')
print(f'Flooded          : {df["flooded"].sum()}')
print(f'Non-flooded      : {(df["flooded"]==0).sum()}')
print(f'Flood rate       : {df["flooded"].mean():.1%}')
print(f'Tiles covered    : {df["tile_id"].nunique()}')
print(f'\nxBD label dist:')
print(df['label'].value_counts())
print(f'\nCrop size stats (pixels):')
df['crop_w'] = df['x2'] - df['x1']
df['crop_h'] = df['y2'] - df['y1']
print(df[['crop_w', 'crop_h']].describe().round(1))

Total crops      : 3986
Flooded          : 1250
Non-flooded      : 2736
Flood rate       : 31.4%
Tiles covered    : 129

xBD label dist:
label
no-damage       2736
major-damage    1250
Name: count, dtype: int64

Crop size stats (pixels):
       crop_w  crop_h
count  3986.0  3986.0
mean    111.3   101.9
std      34.7    17.0
min      64.0    64.0
25%      96.0    96.0
50%      96.0    96.0
75%     124.0   104.0
max     916.0   280.0


## Cell 10 — Run cross-dataset inference

⚠️ **Set `CHECKPOINT_PATH` to your trained model before running.**

In [11]:
CHECKPOINT_PATH = 'C:\\Users\\10211\\Downloads\\new\\models\\best_resnet18_9ch_diff.pth'  # ← UPDATE THIS

macro = run_inference_on_spacenet8_crops(
    sn8_manifest_csv='sn8_manifest.csv',
    checkpoint_path=CHECKPOINT_PATH,
    output_csv='sn8_predictions.csv',
    use_9ch=True,       # True if model uses pre+post+diff (9ch)
    img_size=224,
    batch_size=32,
)

print(f'\n>>> SpaceNet 8 cross-dataset macro-F1: {macro:.4f}')
print('>>> Use this number in your paper under cross-dataset generalisation.')

Device: cuda
Manifest loaded: 3986 building crops
  Flooded    : 1250
  Non-flooded: 2736
Model loaded: C:\Users\10211\Downloads\new\models\best_resnet18_9ch_diff.pth
  640/3986 crops processed...
  1280/3986 crops processed...
  1920/3986 crops processed...
  2560/3986 crops processed...
  3200/3986 crops processed...
  3840/3986 crops processed...

SpaceNet 8 Cross-Dataset Evaluation (Building Crops)
              precision    recall  f1-score   support

 non-flooded      0.694     0.941     0.799      2736
     flooded      0.417     0.092     0.151      1250

    accuracy                          0.675      3986
   macro avg      0.555     0.517     0.475      3986
weighted avg      0.607     0.675     0.596      3986

Macro-F1   : 0.4748
Flooded-F1 : 0.1507

Confusion matrix (rows=true, cols=pred):
[[2575  161]
 [1135  115]]

Expected range: 0.40–0.65 for cross-dataset generalisation
Report this value in your paper regardless — it is a valid finding.

Saved per-crop predictions → 

## Cell 11 — Error analysis (optional)

In [12]:
preds_df = pd.read_csv('sn8_predictions.csv')

print('Per-tile accuracy:')
tile_acc = preds_df.groupby('tile_id')['correct'].mean().sort_values()
print(f'  Worst 5 tiles:')
print(tile_acc.head())
print(f'  Best 5 tiles:')
print(tile_acc.tail())

print(f'\nPrediction distribution:')
print(preds_df['pred_xbd_label'].value_counts())

print(f'\nFalse negatives (flooded but predicted non-flooded):')
fn = preds_df[(preds_df['true_flooded']==1) & (preds_df['pred_flooded']==0)]
print(f'  Count: {len(fn)} / {preds_df["true_flooded"].sum()} flooded buildings')

print(f'\nFalse positives (non-flooded but predicted flooded):')
fp = preds_df[(preds_df['true_flooded']==0) & (preds_df['pred_flooded']==1)]
print(f'  Count: {len(fp)} / {(preds_df["true_flooded"]==0).sum()} non-flooded buildings')

Per-tile accuracy:
  Worst 5 tiles:
tile_id
0_18_69    0.0
0_23_63    0.0
0_22_63    0.0
0_22_62    0.0
0_29_62    0.0
Name: correct, dtype: float64
  Best 5 tiles:
tile_id
0_44_68    1.0
0_44_69    1.0
0_45_67    1.0
0_45_68    1.0
0_45_69    1.0
Name: correct, dtype: float64

Prediction distribution:
pred_xbd_label
no-damage       3398
minor-damage     312
destroyed        231
major-damage      45
Name: count, dtype: int64

False negatives (flooded but predicted non-flooded):
  Count: 1135 / 1250 flooded buildings

False positives (non-flooded but predicted flooded):
  Count: 161 / 2736 non-flooded buildings
